<a href="https://colab.research.google.com/github/sionpardosi/DokterSaku/blob/main/setup_env.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Dokter Saku

In [5]:
from google.colab import userdata
TOKEN = userdata.get("GITHUB_TOKEN")

In [6]:
import os

folders = [
    "src/scraper",
    "src/pipeline",
    "src/guardrails",
    "src/api",
    "eval",
    "data/raw",
    "data/processed",
    "data/golden_queries",
    "notebooks",
    "docs",
    "tests",
    "logs",
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)

# .gitkeep — agar folder kosong tetap ter-track di Git
for folder in ["data/raw", "data/processed", "logs"]:
    open(f"{folder}/.gitkeep", "w").close()

print("✅ Folder structure:")
for f in folders:
    print(f"   📁 {f}")

✅ Folder structure:
   📁 src/scraper
   📁 src/pipeline
   📁 src/guardrails
   📁 src/api
   📁 eval
   📁 data/raw
   📁 data/processed
   📁 data/golden_queries
   📁 notebooks
   📁 docs
   📁 tests
   📁 logs


In [8]:
# ── .gitignore ───────────────────────────────────────────────
with open(".gitignore", "w") as f:
    f.write("""# Secrets
.env
*.key

# Python
__pycache__/
*.py[cod]
.venv/
venv/

# Jupyter
.ipynb_checkpoints/

# Data mentah (terlalu besar)
data/raw/
data/processed/
!data/raw/.gitkeep
!data/processed/.gitkeep

# Golden queries WAJIB di-commit (untuk reproducibility eval)
!data/golden_queries/

# ChromaDB
chroma_db/
*.chroma

# Model weights
models/
*.bin
*.safetensors

# Logs
logs/
!logs/.gitkeep

# OS
.DS_Store
Thumbs.db

# IDE
.vscode/
.idea/
""")

# ── .env.example ─────────────────────────────────────────────
with open(".env.example", "w") as f:
    f.write("""GEMINI_API_KEY=your_gemini_api_key_here
ANTHROPIC_API_KEY=your_anthropic_api_key_here
LLM_PROVIDER=gemini
PORT=8000
""")

print("✅ .gitignore")
print("✅ .env.example")

✅ .gitignore
✅ .env.example


In [9]:
# ── requirements.txt ─────────────────────────────────────────
with open("requirements.txt", "w") as f:
    f.write("""# Dokter Saku — requirements.txt
# Semua versi di-pin untuk reproducibility

# Web Scraping
beautifulsoup4==4.12.3
scrapy==2.11.2
requests==2.31.0
lxml==5.2.1

# Data Processing
pandas==2.2.2
numpy==1.26.4
tqdm==4.66.4

# NLP & Text Processing
spacy==3.7.4
nltk==3.8.1
langdetect==1.0.9

# Embedding & Vector DB
sentence-transformers==3.0.1
chromadb==0.5.3
rank-bm25==0.2.2

# LLM & RAG
langchain==0.2.6
langchain-community==0.2.6
langchain-google-genai==1.0.7
anthropic==0.29.0
google-generativeai==0.7.2

# Evaluation
ragas==0.1.14
datasets==2.20.0

# API & Deploy
fastapi==0.111.1
uvicorn==0.30.1
pydantic==2.7.4
python-dotenv==1.0.1
httpx==0.27.0

# Logging
loguru==0.7.2

# Visualization
matplotlib==3.9.0
seaborn==0.13.2

# Testing
pytest==8.2.2

# Utilities
tenacity==8.3.0
tiktoken==0.7.0
python-multipart==0.0.9
""")

print("✅ requirements.txt")

✅ requirements.txt


In [10]:
# ── config.py ────────────────────────────────────────────────
with open("config.py", "w") as f:
    f.write("""# config.py — Central Configuration Dokter Saku
import os
from dotenv import load_dotenv
load_dotenv()

# API Keys
GEMINI_API_KEY    = os.getenv("GEMINI_API_KEY", "")
ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY", "")

# LLM
LLM_PROVIDER     = os.getenv("LLM_PROVIDER", "gemini")
LLM_MODEL_GEMINI = "gemini-1.5-flash"
LLM_MODEL_CLAUDE = "claude-haiku-4-5-20251001"
LLM_MAX_TOKENS   = 1024
LLM_TEMPERATURE  = 0.1  # rendah = deterministik untuk medical

# Embedding
EMBEDDING_MODEL  = "paraphrase-multilingual-MiniLM-L12-v2"
EMBEDDING_DIM    = 384

# Retrieval
RERANKER_MODEL   = "cross-encoder/ms-marco-MiniLM-L-6-v2"
TOP_K_RETRIEVAL  = 10   # ambil 10, reranker pilih top-3
TOP_K_FINAL      = 3
CHUNK_SIZE       = 512
CHUNK_OVERLAP    = 50

# Safety
CONFIDENCE_THRESHOLD = 0.60  # di bawah ini → safe-refusal
CONFLICT_YEAR_GAP    = 3     # dokumen beda >3 tahun = potensi konflik

# ChromaDB
CHROMA_PERSIST_DIR = "./chroma_db"
CHROMA_COLLECTION  = "dokter_saku_corpus"

# Scraping
SCRAPING_DELAY   = 2.0
SCRAPING_RETRY   = 3
SCRAPING_TIMEOUT = 10
USE_CACHED_DATA  = os.getenv("USE_CACHED_DATA", "false").lower() == "true"

# Evaluation
EVAL_SEED              = 42
GOLDEN_QUERY_PATH      = "./data/golden_queries/golden_queries.json"
TARGET_PRECISION_AT_3  = 0.75
TARGET_FACTUAL_ACC     = 0.80
TARGET_BLOCK_RATE      = 1.00   # 100%
TARGET_PII_REDACTION   = 1.00   # 100%
TARGET_SAFE_REFUSAL    = 0.90
TARGET_LATENCY_P95     = 5.0    # detik

# API Server
API_HOST = "0.0.0.0"
API_PORT = int(os.getenv("PORT", 8000))

# Cost Tracking (Gemini Flash, USD per 1M token)
COST_INPUT_PER_1M  = 0.075
COST_OUTPUT_PER_1M = 0.30

# PII Patterns
PII_PATTERNS = {
    "NIK"   : r"\\b\\d{16}\\b",
    "phone" : r"(\\+62|62|0)[8][0-9]{8,11}",
    "email" : r"[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\\.[a-zA-Z]{2,}",
    "nama"  : None,    # spaCy NER (PERSON entity)
    "alamat": None,    # spaCy NER (GPE/LOC entity)
}

# Triage Keywords — urgent → redirect 119/IGD
TRIAGE_KEYWORDS = [
    "dada sakit", "nyeri dada", "sesak napas", "tidak bisa bernapas",
    "pingsan", "kejang", "stroke", "serangan jantung",
    "overdosis", "keracunan", "pendarahan hebat", "tidak sadarkan diri",
    "bunuh diri", "menyakiti diri", "ingin mati",
]

# Harmful Intent Keywords
HARMFUL_KEYWORDS = [
    "overdosis", "bunuh diri", "cara overdose",
    "dosis mematikan", "cara mati", "racun untuk manusia",
]
""")

print("✅ config.py")

✅ config.py


In [11]:
# ── __init__.py untuk setiap package ─────────────────────────
for pkg in ["src", "src/scraper", "src/pipeline", "src/guardrails", "src/api"]:
    with open(f"{pkg}/__init__.py", "w") as f:
        f.write(f"# Dokter Saku — {pkg}\n")

print("✅ __init__.py semua package")

✅ __init__.py semua package


In [12]:
# ── src/scraper/ ─────────────────────────────────────────────

scrapers = {
    "src/scraper/kemenkes_scraper.py": (
        "Scraper yankes.kemkes.go.id",
        "Target: ~500 dokumen formal medis terverifikasi Kemenkes."
    ),
    "src/scraper/sehatq_scraper.py": (
        "Scraper sehatq.com",
        "Target: ~1.000 artikel kesehatan semi-formal."
    ),
    "src/scraper/bpjs_scraper.py": (
        "Scraper bpjs-kesehatan.go.id",
        "Target: ~200 entri FAQ dan prosedur BPJS."
    ),
    "src/scraper/klikdokter_scraper.py": (
        "Scraper klikdokter.com",
        "Target: ~1.500 Q&A conversational (campur bahasa, informal)."
    ),
}

for filepath, (title, desc) in scrapers.items():
    with open(filepath, "w") as f:
        f.write(f"""\
\"\"\"
{title}
{desc}

CRISP-DM Phase: Data Understanding & Collection
\"\"\"
# TODO: Implement di Step 2
""")

print("✅ src/scraper/ — 4 scraper modules")

✅ src/scraper/ — 4 scraper modules


In [13]:
# ── src/pipeline/ ────────────────────────────────────────────

pipeline_modules = {
    "src/pipeline/preprocessor.py": (
        "Data Preprocessing Pipeline",
        "CRISP-DM Phase 3: Data Preparation\n"
        "Steps: Strip HTML → Normalize → Deduplicate → Chunk (512 token, overlap 50) → Quality Filter\n"
        "Output: clean corpus dengan metadata {source, url, year_published, category}"
    ),
    "src/pipeline/embedder.py": (
        "Embedding + ChromaDB Indexing",
        "Model: paraphrase-multilingual-MiniLM-L12-v2 (dim=384, native bahasa Indonesia)\n"
        "Vector DB: ChromaDB dengan metadata recency untuk conflict detection"
    ),
    "src/pipeline/retriever.py": (
        "Hybrid Retriever",
        "Strategy: BM25 (lexical) + Dense Embedding (semantic) + Cross-encoder Reranker\n"
        "Output: Top-3 dokumen + similarity scores + recency weighting"
    ),
    "src/pipeline/generator.py": (
        "LLM Generation",
        "Provider: Gemini Flash / Claude Haiku (configurable via config.py)\n"
        "Features: system prompt hardening, source citation, cost tracking per query"
    ),
    "src/pipeline/confidence.py": (
        "Confidence Check + Safe-Refusal",
        "Logic:\n"
        "  - similarity < 0.60 → safe-refusal (rujuk ke dokter)\n"
        "  - conflicting docs detected → flag + komunikasikan ke user\n"
        "  - urgent triage detected → redirect 119/IGD"
    ),
}

for filepath, (title, desc) in pipeline_modules.items():
    with open(filepath, "w") as f:
        f.write(f"""\
\"\"\"
{title}
{desc}
\"\"\"
# TODO: Implement di Step 3-4
""")

print("✅ src/pipeline/ — 5 pipeline modules")

✅ src/pipeline/ — 5 pipeline modules


In [14]:
# ── src/guardrails/ ──────────────────────────────────────────

with open("src/guardrails/input_filter.py", "w") as f:
    f.write("""\
\"\"\"
Layer 1: Input Guardrails
Tiga fungsi utama:
  1. Intent Classifier — deteksi harmful intent sebelum retrieval
  2. Prompt Injection Detection — cegah manipulasi system prompt
  3. Triage Detection — kenali gejala darurat, redirect ke 119/IGD
\"\"\"
# TODO: Implement di Step 5
""")

with open("src/guardrails/pii_handler.py", "w") as f:
    f.write("""\
\"\"\"
PII Detection & Redaction
Methods:
  - Regex: NIK (16 digit), nomor HP (+62/0), email
  - spaCy NER: nama (PERSON), alamat (GPE/LOC)
Redaction: ganti dengan placeholder [NAMA_DIREDAKSI], [NIK_DIREDAKSI], dll.
Applied: sebelum retrieval DAN sebelum logging.
\"\"\"
# TODO: Implement di Step 5
""")

with open("src/guardrails/output_filter.py", "w") as f:
    f.write("""\
\"\"\"
Layer 4: Output Guardrails
Post-generation safety check sebelum response dikirim ke user.
Mencegah LLM 'slip' menghasilkan konten berbahaya meski sudah ada input filter.
\"\"\"
# TODO: Implement di Step 5
""")

print("✅ src/guardrails/ — 3 safety modules")

✅ src/guardrails/ — 3 safety modules


In [15]:
# ── src/api/ ─────────────────────────────────────────────────

with open("src/api/models.py", "w") as f:
    f.write("""\
\"\"\"
Pydantic Request & Response Models
Dipakai FastAPI untuk validasi input/output endpoint.
\"\"\"
# TODO: Implement di Step 6
""")

with open("src/api/logger.py", "w") as f:
    f.write("""\
\"\"\"
Audit Logger
Setiap query dicatat: input (PII sudah redacted), retrieved docs, response, latency, cost.
Dipakai untuk: eval, compliance, AI Usage Log di Technical Report.
\"\"\"
# TODO: Implement di Step 6
""")

with open("src/api/main.py", "w") as f:
    f.write("""\
\"\"\"
FastAPI Application — Dokter Saku Inference Endpoint
Routes:
  POST /query  — tanya jawab medis utama
  GET  /health — health check untuk juri
  GET  /docs   — auto-generated Swagger UI
Deploy: Railway (free tier)
\"\"\"
# TODO: Implement di Step 6
""")

print("✅ src/api/ — 3 API modules")

✅ src/api/ — 3 API modules


In [16]:
# ── eval/ ────────────────────────────────────────────────────

eval_modules = {
    "eval/run_eval.py": (
        "Main Eval Script — reproducible",
        "Jalankan semua eval dalam satu script.\n"
        "Usage: python eval/run_eval.py --seed 42\n"
        "Output: eval_results.json dengan semua metric"
    ),
    "eval/ragas_eval.py": (
        "RAGAS Evaluation",
        "Metrics: context_precision (Precision@3), faithfulness (factual accuracy), answer_relevancy\n"
        "Dataset: 50 golden queries dari data/golden_queries/golden_queries.json"
    ),
    "eval/adversarial_eval.py": (
        "Adversarial Test",
        "Test set: 20 prompt injection + 10 harmful query\n"
        "Target: block rate 100%"
    ),
    "eval/pii_eval.py": (
        "PII Redaction Test",
        "Test set: 30 input berisi nama, NIK, alamat\n"
        "Target: redaction rate 100%"
    ),
    "eval/llm_judge.py": (
        "LLM-as-Judge",
        "Medical evaluation criteria: factual accuracy, safety, clarity\n"
        "Dilaporkan: cost per query, reliability diagram"
    ),
}

for filepath, (title, desc) in eval_modules.items():
    with open(filepath, "w") as f:
        f.write(f"""\
\"\"\"
{title}
{desc}
\"\"\"
# TODO: Implement di Step 7
""")

print("✅ eval/ — 5 eval scripts")

✅ eval/ — 5 eval scripts


In [17]:
import json

# ── Golden Queries placeholder ────────────────────────────────
with open("data/golden_queries/golden_queries.json", "w") as f:
    json.dump([], f, indent=2)

with open("data/golden_queries/validation_notes.md", "w") as f:
    f.write("""# Golden Query Validation Notes

## Metodologi
- Total: 50 golden queries
- Dibuat: synthetic via LLM + review manual
- Review manual: minimum 30% sampel (15 query) — sesuai AI Usage Policy
- Kategori: gejala umum, penyakit kronis, obat-obatan, BPJS, darurat

## Status
- [ ] 50 query dibuat (Step 7)
- [ ] 15+ query direview manual
- [ ] Format JSON divalidasi
""")

print("✅ data/golden_queries/")

✅ data/golden_queries/


In [18]:
# ── README.md ─────────────────────────────────────────────────
with open("README.md", "w") as f:
    f.write("""# Dokter Saku 🩺
**Asisten Kesehatan Cerdas Berbasis Data Klinis Indonesia**

> INaAI Competition 2026 × IT Del | AI Engineer Track | Domain 3: Medical AI Health

---

## Overview
Sistem tanya-jawab medis berbahasa Indonesia yang menjawab berdasarkan sumber
klinis terverifikasi, dengan safe-refusal aktif untuk pertanyaan di luar batas
kepercayaan sistem.

**Tiga pilar utama:**
- **RAG Pipeline** — Hybrid retrieval: BM25 + Dense + Reranker + ChromaDB
- **Guardrails Medical** — 4-layer safety architecture
- **PII Handling** — Redaksi otomatis nama, NIK, alamat

---

## System Architecture

```
USER INPUT
    │
[Layer 1] Input Guardrails + Intent Classifier
          → Harmful?  → BLOCK
          → PII?      → REDACT
          → Injection?→ BLOCK
    │
[Layer 2] Hybrid Retriever
          → BM25 + Dense Embedding + Reranker
          → Top-3 dokumen + recency weighting
    │
[Layer 3] Confidence Check
          → score < 0.60? → Safe-Refusal → rujuk dokter
          → Urgent?       → Redirect 119/IGD
    │
[Layer 4] LLM Generation + Output Guardrails
          → System prompt hardening
          → Response + sitasi sumber
    │
  RESPONSE
```

---

## Target Metrics

| Metric                | Target  |
|-----------------------|---------|
| Retrieval Precision@3 | ≥ 0.75  |
| Factual Accuracy      | ≥ 0.80  |
| Guardrails Block Rate | 100%    |
| PII Redaction Rate    | 100%    |
| Safe-Refusal Rate     | ≥ 90%   |
| Latency p95           | ≤ 5 dtk |

---

## Data Sources

| Sumber                  | Tipe          | Volume      |
|-------------------------|---------------|-------------|
| yankes.kemkes.go.id     | Formal        | ~500 dok    |
| sehatq.com              | Semi-formal   | ~1.000 dok  |
| bpjs-kesehatan.go.id    | Formal        | ~200 entri  |
| klikdokter.com          | Conversational| ~1.500 Q&A  |

---

## Tech Stack

| Komponen    | Tool                                    |
|-------------|-----------------------------------------|
| Scraping    | BeautifulSoup, Scrapy                   |
| Processing  | Pandas, spaCy, NLTK                     |
| Embedding   | paraphrase-multilingual-MiniLM-L12-v2   |
| Vector DB   | ChromaDB                                |
| Retrieval   | rank-bm25 + dense + cross-encoder       |
| LLM         | Gemini Flash / Claude Haiku             |
| Evaluation  | RAGAS                                   |
| API         | FastAPI + Railway                       |

---

## Setup
```bash
pip install -r requirements.txt
cp .env.example .env
uvicorn src.api.main:app --reload
```

## Eval (reproducible)
```bash
python eval/run_eval.py --seed 42
```

---

## Skenario 3 — Design Decision

> *PM: "Sistem tidak boleh pernah bilang saya tidak tahu"*

Dokter Saku mengimplementasi **informed refusal**:
- confidence < 0.60 → konteks umum + redirect ke tenaga medis
- query urgent → langsung redirect ke 119/IGD
- *"Saya tidak tahu, konsultasikan dokter"* adalah **fitur keselamatan**, bukan kelemahan.
""")

print("✅ README.md")

✅ README.md


In [19]:
import os

expected = [
    "README.md", "requirements.txt", "config.py", ".gitignore", ".env.example",
    "src/__init__.py",
    "src/scraper/__init__.py", "src/scraper/kemenkes_scraper.py",
    "src/scraper/sehatq_scraper.py", "src/scraper/bpjs_scraper.py",
    "src/scraper/klikdokter_scraper.py",
    "src/pipeline/__init__.py", "src/pipeline/preprocessor.py",
    "src/pipeline/embedder.py", "src/pipeline/retriever.py",
    "src/pipeline/generator.py", "src/pipeline/confidence.py",
    "src/guardrails/__init__.py", "src/guardrails/input_filter.py",
    "src/guardrails/pii_handler.py", "src/guardrails/output_filter.py",
    "src/api/__init__.py", "src/api/main.py",
    "src/api/models.py", "src/api/logger.py",
    "eval/run_eval.py", "eval/ragas_eval.py", "eval/adversarial_eval.py",
    "eval/pii_eval.py", "eval/llm_judge.py",
    "data/golden_queries/golden_queries.json",
    "data/golden_queries/validation_notes.md",
]

print("=" * 50)
print(" 🩺 Dokter Saku — File Check")
print("=" * 50)

all_ok = True
for f in expected:
    exists = os.path.exists(f)
    print(f"  {'✅' if exists else '❌'}  {f}")
    if not exists:
        all_ok = False

print("=" * 50)
print(f"  {'🎉 Semua file siap!' if all_ok else '⚠️  Ada yang kurang!'}")
print("=" * 50)

 🩺 Dokter Saku — File Check
  ✅  README.md
  ✅  requirements.txt
  ✅  config.py
  ✅  .gitignore
  ✅  .env.example
  ✅  src/__init__.py
  ✅  src/scraper/__init__.py
  ✅  src/scraper/kemenkes_scraper.py
  ✅  src/scraper/sehatq_scraper.py
  ✅  src/scraper/bpjs_scraper.py
  ✅  src/scraper/klikdokter_scraper.py
  ✅  src/pipeline/__init__.py
  ✅  src/pipeline/preprocessor.py
  ✅  src/pipeline/embedder.py
  ✅  src/pipeline/retriever.py
  ✅  src/pipeline/generator.py
  ✅  src/pipeline/confidence.py
  ✅  src/guardrails/__init__.py
  ✅  src/guardrails/input_filter.py
  ✅  src/guardrails/pii_handler.py
  ✅  src/guardrails/output_filter.py
  ✅  src/api/__init__.py
  ✅  src/api/main.py
  ✅  src/api/models.py
  ✅  src/api/logger.py
  ✅  eval/run_eval.py
  ✅  eval/ragas_eval.py
  ✅  eval/adversarial_eval.py
  ✅  eval/pii_eval.py
  ✅  eval/llm_judge.py
  ✅  data/golden_queries/golden_queries.json
  ✅  data/golden_queries/validation_notes.md
  🎉 Semua file siap!


In [20]:
!git add .
!git status --short

fatal: not a git repository (or any of the parent directories): .git
fatal: not a git repository (or any of the parent directories): .git


In [21]:
!git commit -m "feat: initial project structure DokterSaku

- Setup folder: src, eval, data, notebooks, docs, tests
- Add: README, requirements.txt, config.py, .gitignore
- Add scrapers: kemenkes, sehatq, bpjs, klikdokter (placeholder)
- Add pipeline: preprocessor, embedder, retriever, generator, confidence
- Add guardrails: input_filter, pii_handler, output_filter
- Add api: main, models, logger
- Add eval: run_eval, ragas, adversarial, pii, llm_judge
- INaAI Competition 2026 x IT Del | Domain 3: Medical AI Health"

!git push origin main

print()
print("=" * 50)
print("  ✅ STEP 1 SELESAI & PUSHED!")
print("  🔗 github.com/sionpardosi/DokterSaku")
print("=" * 50)
print("  ➡️  Lanjut: Step 2 — Data Pipeline (Scraping)")

SyntaxError: unterminated string literal (detected at line 10) (2023988772.py, line 10)